# Treino U-Net no Kaggle (GPU)

Pré-treino (Inria e/ou RoofSat, binário) + fine-tuning (chips multiclasse). **Tudo automático.**

**Só precisas de:**
1. **Settings → Accelerator:** GPU | **Internet:** On
2. **Add data:** anexar os datasets (gerados por `scripts/prepare_kaggle_zips.py`): **roof-chips-multiclass** (obrigatório), **roof-inria-patches** e **roof-roofsat** (opcionais).
3. **Run All** — os caminhos são detetados sozinhos; o pré-treino corre para Inria e/ou RoofSat se anexados; o fine-tuning usa o melhor checkpoint disponível.

In [ ]:
# Verificar GPU
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', out.stdout.strip() if out.returncode == 0 else 'Nenhuma GPU')

In [ ]:
# Clone do repositório roofArea
import os
REPO_URL = "https://github.com/phmotad/roofArea.git"
REPO_DIR = "roofArea"

if os.path.isdir(REPO_DIR) and os.path.isfile(os.path.join(REPO_DIR, "scripts", "train_unet.py")):
    print("Projeto já presente.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
os.chdir(REPO_DIR)
print("Diretório atual:", os.getcwd())

In [ ]:
# Instalar dependências
subprocess.run(["pip", "install", "-q", "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
subprocess.run(["pip", "install", "-q", "pillow", "numpy", "opencv-python-headless", "scikit-image", "huggingface_hub"], check=True)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

**Caminhos:** Detetados automaticamente em `/kaggle/input/`. Não precisas de editar nada.

In [ ]:
# Caminhos — deteção automática dos datasets em /kaggle/input/
import os
KAGGLE_INPUT = "/kaggle/input"

def _has_subdirs(root, *subdirs):
    return all(os.path.isdir(os.path.join(root, d)) for d in subdirs)

def _has_files_in(dirpath, ext=".png"):
    if not os.path.isdir(dirpath):
        return False
    try:
        return any(f.lower().endswith(ext) for f in os.listdir(dirpath)[:5])
    except OSError:
        return False

CHIPS_PATH = ""
INRIA_PATH = ""
ROOFSAT_PATH = ""
try:
    slugs = [s for s in os.listdir(KAGGLE_INPUT) if os.path.isdir(os.path.join(KAGGLE_INPUT, s))]
except OSError:
    slugs = []

# Preferir por nome do slug (roof-chips-multiclass, roof-inria-patches, roof-roofsat)
for slug in slugs:
    root = os.path.join(KAGGLE_INPUT, slug)
    rs = os.path.join(root, "Roofsat") if os.path.isdir(os.path.join(root, "Roofsat")) else root
    if ("roofsat" in slug.lower() and os.path.isdir(os.path.join(rs, "building_masks"))
        and os.path.isfile(os.path.join(rs, "train.txt"))
        and (os.path.isdir(os.path.join(rs, "img_color")) or os.path.isdir(os.path.join(rs, "img")))):
        ROOFSAT_PATH = rs
    cand = os.path.join(root, "chips_multiclass") if os.path.isdir(os.path.join(root, "chips_multiclass")) else root
    if ("chip" in slug.lower() or "multiclass" in slug.lower()) and _has_subdirs(cand, "images", "masks"):
        CHIPS_PATH = cand
    inria_cand = os.path.join(root, "dados_inria") if os.path.isdir(os.path.join(root, "dados_inria")) else root
    if "inria" in slug.lower() and _has_subdirs(inria_cand, "images", "masks") and _has_files_in(os.path.join(inria_cand, "images")):
        INRIA_PATH = inria_cand

# Preencher por estrutura o que faltar
for slug in slugs:
    root = os.path.join(KAGGLE_INPUT, slug)
    if not ROOFSAT_PATH:
        rs = os.path.join(root, "Roofsat") if os.path.isdir(os.path.join(root, "Roofsat")) else root
        if os.path.isdir(os.path.join(rs, "building_masks")) and os.path.isfile(os.path.join(rs, "train.txt")):
            if os.path.isdir(os.path.join(rs, "img_color")) or os.path.isdir(os.path.join(rs, "img")):
                ROOFSAT_PATH = rs
    if not CHIPS_PATH:
        cand = os.path.join(root, "chips_multiclass") if os.path.isdir(os.path.join(root, "chips_multiclass")) else root
        if _has_subdirs(cand, "images", "masks") and _has_files_in(os.path.join(cand, "images")):
            CHIPS_PATH = cand
    if not INRIA_PATH:
        inria_cand = os.path.join(root, "dados_inria") if os.path.isdir(os.path.join(root, "dados_inria")) else root
        if _has_subdirs(inria_cand, "images", "masks") and _has_files_in(os.path.join(inria_cand, "images")):
            if inria_cand != CHIPS_PATH:
                INRIA_PATH = inria_cand

if not CHIPS_PATH:
    raise SystemExit("Nenhum dataset de chips em /kaggle/input/. Anexa roof-chips-multiclass (prepare_kaggle_zips).")
print("Chips:", CHIPS_PATH)
if INRIA_PATH:
    print("Inria: anexado →", INRIA_PATH)
else:
    INRIA_PATH = "/kaggle/working/dados_inria"
    print("Inria: será descarregado →", INRIA_PATH)
if ROOFSAT_PATH:
    print("RoofSat: anexado →", ROOFSAT_PATH)
else:
    print("RoofSat: não anexado")

In [ ]:
# Descarregar Inria só se não tiver images/ e masks/ (ex.: quando não anexaste roof-inria-patches)
import os
import subprocess
if not os.path.isdir(os.path.join(INRIA_PATH, "images")) or not os.path.isdir(os.path.join(INRIA_PATH, "masks")):
    print("A descarregar Inria para", INRIA_PATH, "...")
    subprocess.run(["python", "-m", "scripts.download_inria_dataset", "--output_dir", INRIA_PATH], check=True)
else:
    print("Inria já disponível em", INRIA_PATH)

In [ ]:
# Pré-treino (binário, Inria) — só corre se existir images/ e masks/ em INRIA_PATH
import subprocess
import os
os.makedirs("/kaggle/working/models", exist_ok=True)
if os.path.isdir(os.path.join(INRIA_PATH, "images")) and os.path.isdir(os.path.join(INRIA_PATH, "masks")):
    subprocess.run([
        "python", "-u", "-m", "scripts.train_unet",
        "--data_dir", INRIA_PATH,
        "--output", "/kaggle/working/models/unet_roof_pretrain.pt",
        "--num_classes", "1", "--size", "512", "512", "--epochs", "30", "--device", "cuda"
    ], check=True)
else:
    print("Inria sem images/masks; a ignorar pré-treino Inria.")

**Pré-treino RoofSat (opcional):** Só corre se anexaste o dataset `roof-roofsat`. Usa `train.txt`/`val.txt` e guarda `unet_roof_roofsat_pretrain.pt`. Podes usar este checkpoint em vez do Inria no fine-tuning (altera `--pretrain` na célula seguinte).

In [ ]:
# Pré-treino RoofSat (opcional) — só corre se ROOFSAT_PATH existir
import subprocess
import os
if ROOFSAT_PATH and os.path.isdir(os.path.join(ROOFSAT_PATH, "building_masks")):
    os.makedirs("/kaggle/working/models", exist_ok=True)
    subprocess.run([
        "python", "-u", "-m", "scripts.train_unet",
        "--data_dir", ROOFSAT_PATH,
        "--dataset_type", "roofsat",
        "--output", "/kaggle/working/models/unet_roof_roofsat_pretrain.pt",
        "--num_classes", "1", "--size", "512", "512", "--epochs", "30", "--device", "cuda"
    ], check=True)
else:
    print("RoofSat não disponível; a ignorar pré-treino RoofSat.")

In [ ]:
# Fine-tuning (multiclasse, chips) — usa pré-treino Inria ou RoofSat se existir; senão treina só com chips
import subprocess
import os
PRETRAIN_PATH = "/kaggle/working/models/unet_roof_pretrain.pt"
if not os.path.isfile(PRETRAIN_PATH):
    PRETRAIN_PATH = "/kaggle/working/models/unet_roof_roofsat_pretrain.pt"
cmd = [
    "python", "-u", "-m", "scripts.train_unet",
    "--data_dir", CHIPS_PATH,
    "--output", "/kaggle/working/models/unet_roof_multiclass.pt",
    "--num_classes", "5", "--size", "512", "512", "--epochs", "30", "--device", "cuda"
]
if os.path.isfile(PRETRAIN_PATH):
    cmd.extend(["--pretrain", PRETRAIN_PATH])
    print("Fine-tuning com pretrain:", PRETRAIN_PATH)
else:
    print("Fine-tuning sem pretrain (só chips).")
subprocess.run(cmd, check=True)

**Guardar o modelo:** Save Version (canto superior direito) com **Save output** ativado. O ficheiro `unet_roof_multiclass.pt` ficará em Output para download.

In [ ]:
# Listar modelos em /kaggle/working/models
import os
for f in sorted(os.listdir("/kaggle/working/models")):
    path = os.path.join("/kaggle/working/models", f)
    print(f, "—", os.path.getsize(path) // 1024, "KB")